In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHUNK = 5_000_000
cols = ["user_id", "item_id", "category_id", "behavior_type"]
data_path = Path.cwd().parent / 'UserBehavior_cleaning.csv'

chunks = []
for c in pd.read_csv(
    data_path,
    header = 0,
    usecols=cols,
    chunksize=CHUNK,
    dtype={
        "user_id": "int32",
        "item_id": "int32",
        "category_id": "int32",
        "behavior_type": "category"
    },
):
    chunks.append(c)

df = pd.concat(chunks, ignore_index=True)

print(f"总记录数: {len(df):,}")
print(f"\n行为分布:\n{df['behavior_type'].value_counts()}")

总记录数: 86,870,392

行为分布:
behavior_type
pv      77887714
cart     4752175
fav      2479677
buy      1750826
Name: count, dtype: int64


In [10]:
# 以次数为基础构建品类宽表
cat_pivot = (
    df.groupby(["category_id", "behavior_type"],observed=False)
    .size()
    .unstack(fill_value=0)
)
for col in ["pv", "fav", "cart", "buy"]:
    if col not in cat_pivot.columns:
        cat_pivot[col] = 0

# GMV 代理分：购买权重最高，加购次之
cat_pivot["gmv_proxy"]  = cat_pivot["buy"] * 3 + cat_pivot["cart"] * 1
# 浏览 → 购买 转化率（次数口径）
cat_pivot["pv_to_buy"]  = cat_pivot["buy"]  / cat_pivot["pv"].replace(0, np.nan)
# 加购 → 购买 转化率（次数口径）
cat_pivot["cart_to_buy"] = cat_pivot["buy"] / cat_pivot["cart"].replace(0, np.nan)
# 收藏+加购合并 → 购买 转化率（次数口径）
cat_pivot["fav_cart_sum"]        = cat_pivot["fav"] + cat_pivot["cart"]
cat_pivot["fav_cart_to_buy_rate"] = cat_pivot["buy"] / cat_pivot["fav_cart_sum"].replace(0, np.nan)

print(cat_pivot.head(3))

behavior_type  buy  cart  fav    pv  gmv_proxy  pv_to_buy  cart_to_buy  \
category_id                                                              
80               0     0    0    17          0   0.000000          NaN   
1147             0     0    0     1          0   0.000000          NaN   
2171            90   127   42  1358        397   0.066274     0.708661   

behavior_type  fav_cart_sum  fav_cart_to_buy_rate  
category_id                                        
80                        0                   NaN  
1147                      0                   NaN  
2171                    169              0.532544  


In [3]:
# ── 用户级：每品类各行为的去重用户数 ──
cat_user = (
    df.groupby(["category_id", "behavior_type"],observed = False)["user_id"]
    .apply(set)
    .unstack(level = 'behavior_type')
    .rename(columns = lambda x: f"has_{x}")
)
for col in cat_user.columns:
    cat_user[col] = cat_user[col].apply(lambda x: x if isinstance(x, set) else set())

cat_user['cart_to_buy_users'] = cat_user.apply(lambda row: len(row['has_cart'] & row['has_buy']),axis = 1)
cat_user['cart_users'] = cat_user['has_cart'].apply(len)
cat_user['fav_cart_users'] = cat_user.apply(lambda row: len(row['has_fav'] | row['has_cart']), axis = 1)
cat_user['fav_cart_to_buy_users'] = cat_user.apply(lambda row: len((row['has_fav'] | row['has_cart']) & row['has_buy']), axis = 1)


# 客户口径转化率
cat_user["cart_to_buy_users_rate"]    = cat_user["cart_to_buy_users"]  / cat_user["cart_users"].replace(0, np.nan)
cat_user["fav_cart_to_buy_users_rate"] = (
    cat_user["fav_cart_to_buy_users"]
    / (cat_user["fav_cart_users"]).replace(0, np.nan)
)

#删除集合列
set_cols = [col for col in cat_user.columns if cat_user[col].apply(lambda x: isinstance(x, set)).any()]
cat_user = cat_user.drop(columns=set_cols)

# 合并两张宽表
cat_full = cat_pivot.join(cat_user)
print(cat_full.shape)
cat_full.head(10)

(9377, 15)


behavior_type,buy,cart,fav,pv,gmv_proxy,pv_to_buy,cart_to_buy,fav_cart_sum,favcart_to_buy_rate,cart_to_buy_users,cart_users,fav_cart_users,fav_cart_to_buy_users,cart_to_buy_users_rate,fav_cart_to_buy_users_rate
category_id,,,,,,,,,,,,,,,
80,0,0,0,17,0,0.000000,NaN,0,NaN,0,0,0,0,NaN,NaN
1147,0,0,0,1,0,0.000000,NaN,0,NaN,0,0,0,0,NaN,NaN
2171,90,127,42,1358,397,0.066274,0.708661,169,0.532544,16,110,146,18,0.145455,0.123288
2410,14,33,12,581,75,0.024096,0.424242,45,0.311111,2,31,42,2,0.064516,0.047619
2424,4,1,0,44,13,0.090909,4.000000,1,4.000000,1,1,1,1,1.000000,1.000000
2818,11,1,0,20,34,0.550000,11.000000,1,11.000000,0,1,1,0,0.000000,0.000000
3579,8,8,0,68,32,0.117647,1.000000,8,1.000000,0,6,6,0,0.000000,0.000000
4907,23,41,7,462,110,0.049784,0.560976,48,0.479167,6,38,44,6,0.157895,0.136364
5064,241,1288,1309,21101,2011,0.011421,0.187112,2597,0.092799,62,838,1562,81,0.073986,0.051857


In [4]:
top_gmv = cat_full.nlargest(20, "gmv_proxy").reset_index()

total_buy  = cat_full["buy"].sum()
top5_buy   = top_gmv["buy"].head(5).sum()
top10_buy  = top_gmv["buy"].head(10).sum()

print("【品类 GMV 驱动 Top10】")
print(top_gmv[["category_id","pv","fav","cart","buy","gmv_proxy","pv_to_buy"]].head(10).to_string(index=False))
print(f"\nTop5  品类购买占全平台: {top5_buy/total_buy:.1%}")
print(f"Top10 品类购买占全平台: {top10_buy/total_buy:.1%}")

【品类 GMV 驱动 Top10】
 category_id      pv    fav   cart   buy  gmv_proxy  pv_to_buy
     4756105 3883924 119738 185099 24363     258188   0.006273
     4145813 2749173  95786 150563 27593     233342   0.010037
      982926 2428560  76583 132227 21601     197030   0.008895
     4801426 1622540  58675 104830 23110     174160   0.014243
     2735466  977228  30978  81197 29476     169625   0.030163
     1464116  597584  21476  52364 30034     142466   0.050259
     2355072 2756514  75526 105261 10903     137970   0.003955
     2885642  826298  24903  48332 27206     129950   0.032925
     3607361 2583743  61681  94205 11191     127778   0.004331
     1320293 1564218  46369  81708 14950     126558   0.009557

Top5  品类购买占全平台: 7.2%
Top10 品类购买占全平台: 12.6%


In [12]:
# ── 过滤噪声：cart≥200 且 buy≥50 ──
CART_MIN = 200
BUY_MIN  = 50

cat_conv = cat_pivot[
    (cat_pivot["cart"] >= CART_MIN) &
    (cat_pivot["buy"]  >= BUY_MIN)
].copy()

# ── 次数口径：加购→购买 ──
price_sensitive_cnt = cat_conv.nsmallest(20, "cart_to_buy").reset_index()
impulse_cnt         = cat_conv.nlargest(20,  "cart_to_buy").reset_index()

# ── 次数口径：收藏+加购→购买 ──
price_sensitive_fc  = cat_conv.nsmallest(20, "fav_cart_to_buy_rate").reset_index()
impulse_fc          = cat_conv.nlargest(20,  "fav_cart_to_buy_rate").reset_index()

# ── 客户口径：加购→购买（去重用户） ──
cat_conv_u = cat_full[
    (cat_full["cart_users"] >= 100) &
    (cat_full["cart_to_buy_users"]  >= 30)
].copy()
price_sensitive_u = cat_conv_u.nsmallest(20, "cart_to_buy_users_rate").reset_index()
impulse_u         = cat_conv_u.nlargest(20,  "cart_to_buy_users_rate").reset_index()

print("=" * 60)
print("▶ 价格敏感品类 Top10（cart→buy 次数口径，转化率最低）")
print(price_sensitive_cnt[["category_id","cart","buy","cart_to_buy"]].head(10).to_string(index=False))

print("\n▶ 价格敏感品类 Top10（fav+cart→buy 次数口径，转化率最低）")
print(price_sensitive_fc[["category_id","fav","cart","buy","fav_cart_to_buy_rate"]].head(10).to_string(index=False))

print("\n▶ 价格敏感品类 Top10（cart→buy 客户口径，转化率最低）")
print(price_sensitive_u[["category_id","cart_users","cart_to_buy_users","cart_to_buy_users_rate"]].head(10).to_string(index=False))

print("\n▶ 高意愿/冲动消费品类 Top10（fav+cart→buy 次数口径，转化率最高）")
print(impulse_fc[["category_id","cart","fav","buy","fav_cart_to_buy_rate"]].head(10).to_string(index=False))

print("\n▶ 高意愿/冲动消费品类 Top10（cart→buy 客户口径，转化率最高）")
print(impulse_u[["category_id","cart_users","cart_to_buy_users","cart_to_buy_users_rate"]].head(10).to_string(index=False))

▶ 价格敏感品类 Top10（cart→buy 次数口径，转化率最低）
 category_id  cart  buy  cart_to_buy
     4655941  1131   55     0.048630
     2352202  1369   79     0.057706
     3457718   942   61     0.064756
     1202097   938   78     0.083156
     2421366   608   51     0.083882
      154040 25296 2176     0.086022
     4825728   661   57     0.086233
     4358294  2045  177     0.086553
     4466876  1057   93     0.087985
     4509358  1975  177     0.089620

▶ 价格敏感品类 Top10（fav+cart→buy 次数口径，转化率最低）
 category_id  fav  cart  buy  fav_cart_to_buy_rate
     4655941  940  1131   55              0.026557
     2352202 1252  1369   79              0.030141
     3457718  704   942   61              0.037060
     4358294 2525  2045  177              0.038731
     4825728  686   661   57              0.042316
       22129  596   557   50              0.043365
     1202097  777   938   78              0.045481
     4466876  941  1057   93              0.046547
     4509358 1731  1975  177              0.047760
     3

In [22]:
# ── 只取需要的行为 ──
df_interest = df[df["behavior_type"].isin(["fav", "cart", "buy"])].copy()
df_interest["is_fav_cart"] = df_interest["behavior_type"].isin(["fav", "cart"])
df_interest["is_buy"]      = df_interest["behavior_type"] == "buy"

# ── 每个 (item_id, user_id) 是否有过 fav/cart、是否有过 buy ──
user_item = (
    df_interest
    .groupby(["item_id", "user_id"], observed=True)
    .agg(has_fav_cart=("is_fav_cart", "any"),
         has_buy      =("is_buy",       "any"))
)

# ── 商品级聚合 ──
item_user_stats = user_item.groupby("item_id").agg(
    fav_cart_users=("has_fav_cart", "sum"),
    buy_users      =("has_buy",      "sum")
)

# ── 有路径保证：fav/cart 且 buy 的用户数 ──
item_user_stats["fav_cart_to_buy_users"] = (
    user_item[user_item["has_fav_cart"] & user_item["has_buy"]]
    .groupby("item_id")
    .size()
    .reindex(item_user_stats.index, fill_value=0)
)

# ── 合并次数宽表 ──
item_pivot = (
    df.groupby(["item_id", "behavior_type"], observed=True)
    .size()
    .unstack(fill_value=0)
)
for col in ["pv", "fav", "cart", "buy"]:
    if col not in item_pivot.columns:
        item_pivot[col] = 0

item_full = item_pivot.join(item_user_stats)

PV_MIN = 50
item_f = item_full[item_full["pv"] >= PV_MIN].copy()

item_f["heat_score"]               = item_f["cart"] * 2 + item_f["fav"] * 1
item_f["potential_ratio"]          = item_f["heat_score"] / (item_f["buy"] + 1)
item_f["fav_cart_to_buy_rate"]     = (
    item_f["fav_cart_to_buy_users"] / item_f["fav_cart_users"].replace(0, np.nan)
)

heat_q75 = item_f["heat_score"].quantile(0.75)
item_hot = item_f[item_f["heat_score"] >= heat_q75].copy()
item_hot_filtered = item_hot[item_hot['buy'] > 0].copy()

sleeper_hits = (
    item_hot_filtered
    .nlargest(30, "potential_ratio")
    .reset_index()
    [["item_id","pv","fav","cart","buy",
      "fav_cart_users","fav_cart_to_buy_users",
      "heat_score","potential_ratio","fav_cart_to_buy_rate"]]
)

print("▶ 蓄力型爆款潜力品 Top20")
print(sleeper_hits.head(20).to_string(index=False))

▶ 蓄力型爆款潜力品 Top20
 item_id   pv  fav  cart  buy  fav_cart_users  fav_cart_to_buy_users  heat_score  potential_ratio  fav_cart_to_buy_rate
 2671601 5118  158   173    1           329.0                    0.0         504           252.00              0.000000
 2161188 7566  233   117    1           349.0                    0.0         467           233.50              0.000000
 2678768 7035  194    88    1           280.0                    0.0         370           185.00              0.000000
 2642113 6319  214   230    3           441.0                    1.0         674           168.50              0.002268
 1283005 4025  199   190    3           373.0                    0.0         579           144.75              0.000000
 2809893 1331  103    92    1           193.0                    0.0         287           143.50              0.000000
  815347 1089   88    99    1           184.0                    0.0         286           143.00              0.000000
  107385 1699   67    9

In [ ]:
PALETTE = {
    "bg":    "#0f1117", "panel":  "#1a1d27",
    "a1":    "#ff6b35", "a2":     "#00d4aa",
    "a3":    "#7b61ff", "a4":     "#ffcc00",
    "text":  "#e8eaf0", "muted":  "#6b7280", "grid": "#2a2d3a",
}

def style_ax(ax, title, fontsize=11):
    ax.set_facecolor(PALETTE["panel"])
    ax.spines[:].set_color(PALETTE["grid"])
    ax.tick_params(colors=PALETTE["muted"], labelsize=8)
    ax.xaxis.label.set_color(PALETTE["muted"])
    ax.yaxis.label.set_color(PALETTE["muted"])
    ax.set_title(title, color=PALETTE["text"], fontsize=fontsize,
                 fontweight="bold", pad=10)
    ax.grid(axis="x", color=PALETTE["grid"], linewidth=0.5, alpha=0.6)

fig = plt.figure(figsize=(22, 26), facecolor=PALETTE["bg"])
fig.suptitle("商品与品类分析  ·  UserBehavior 数据集",
             fontsize=22, fontweight="bold", color=PALETTE["text"], y=0.99)

gs = GridSpec(3, 2, figure=fig, hspace=0.48, wspace=0.35,
              top=0.96, bottom=0.04, left=0.06, right=0.97)

# ── 图1: Top20 品类 GMV 代理分 ──
ax1 = fig.add_subplot(gs[0, :])
colors1 = [PALETTE["a1"] if i < 3 else PALETTE["a3"] for i in range(len(top_gmv))]
bars = ax1.barh(top_gmv["category_id"].astype(str), top_gmv["gmv_proxy"],
                color=colors1, edgecolor="none", height=0.65)
for bar, val in zip(bars, top_gmv["gmv_proxy"]):
    ax1.text(bar.get_width() * 1.005, bar.get_y() + bar.get_height()/2,
             f"{val:,.0f}", va="center", ha="left", color=PALETTE["muted"], fontsize=7)
style_ax(ax1, "① 品类 GMV 贡献 Top20（代理分 = buy×3 + cart×1）", fontsize=12)
ax1.invert_yaxis()
ax1.set_xlabel("GMV 代理分")
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e4:.0f}万"))
for i, row in top_gmv.head(3).iterrows():
    label = ["🥇", "🥈", "🥉"][i]
    ax1.text(top_gmv["gmv_proxy"].max() * 0.002, i, f" {label} 核心品类",
             va="center", color=PALETTE["a1"], fontsize=8)

# ── 图2: 价格敏感品类（fav+cart→buy 次数口径，转化率最低 Top15）──
ax2 = fig.add_subplot(gs[1, 0])
top_ps = price_sensitive_fc.head(15)
bars2 = ax2.barh(top_ps["category_id"].astype(str), top_ps["fav_cart_to_buy_rate"],
                 color=PALETTE["a2"], edgecolor="none", height=0.65)
for bar, val in zip(bars2, top_ps["fav_cart_to_buy_rate"]):
    ax2.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", ha="left", color=PALETTE["muted"], fontsize=7)
ax2.axvline(top_ps["fav_cart_to_buy_rate"].mean(), color=PALETTE["a1"],
            linestyle="--", linewidth=1, alpha=0.8, label=f"均值={top_ps['fav_cart_to_buy_rate'].mean():.3f}")
ax2.legend(fontsize=7, labelcolor=PALETTE["muted"], framealpha=0)
style_ax(ax2, "② 价格敏感品类 Top15\n（收藏、加购→购买 次数转化率最低）")
ax2.invert_yaxis()
ax2.set_xlabel("fav+cart → buy 转化率（次数口径）")

# ── 图3: 价格敏感品类（客户口径，fav/cart→buy 用户转化率最低 Top15）──
ax3 = fig.add_subplot(gs[1, 1])
top_psu = price_sensitive_u.head(15)
bars3 = ax3.barh(top_psu["category_id"].astype(str), top_psu["fav_cart_to_buy_users_rate"],
                 color=PALETTE["a4"], edgecolor="none", height=0.65)
for bar, val in zip(bars3, top_psu["fav_cart_to_buy_users_rate"]):
    ax3.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", ha="left", color=PALETTE["muted"], fontsize=7)
ax3.axvline(top_psu["fav_cart_to_buy_users_rate"].mean(), color=PALETTE["a1"],
            linestyle="--", linewidth=1, alpha=0.8,
            label=f"均值={top_psu['fav_cart_to_buy_users_rate'].mean():.3f}")
ax3.legend(fontsize=7, labelcolor=PALETTE["muted"], framealpha=0)
style_ax(ax3, "③ 价格敏感品类 Top15\n（加购→购买 客户转化率最低）")
ax3.invert_yaxis()
ax3.set_xlabel("cart → buy 转化率（客户口径）")

# ── 图4: 蓄力型爆款 — 热度 vs 购买量散点图 ──
ax4 = fig.add_subplot(gs[2, 0])
ax4.set_facecolor(PALETTE["panel"])
ax4.spines[:].set_color(PALETTE["grid"])
plot_items = sleeper_hits.head(50)
sc = ax4.scatter(plot_items["heat_score"], plot_items["buy"],
                 c=plot_items["potential_ratio"], cmap="plasma",
                 s=80, alpha=0.85, edgecolors="none")
for _, row in sleeper_hits.head(10).iterrows():
    ax4.annotate(str(row["item_id"]), (row["heat_score"], row["buy"]),
                 fontsize=6, color=PALETTE["muted"],
                 xytext=(5, 3), textcoords="offset points")
cb = plt.colorbar(sc, ax=ax4)
cb.ax.tick_params(colors=PALETTE["muted"], labelsize=7)
cb.set_label("潜力比率", color=PALETTE["muted"], fontsize=8)
ax4.set_xlabel("热度分（cart×2 + fav×1）")
ax4.set_ylabel("购买次数")
ax4.tick_params(colors=PALETTE["muted"], labelsize=8)
ax4.grid(color=PALETTE["grid"], linewidth=0.5, alpha=0.5)
ax4.set_title("④ 蓄力型爆款潜力品\n（热度高、购买尚未爆发 → 右下角为目标区域）",
              color=PALETTE["text"], fontsize=11, fontweight="bold", pad=10)

# ── 图5: 蓄力型爆款 Top15 — 客户口径收藏、加购 vs 购买对比柱状图 ──
ax5 = fig.add_subplot(gs[2, 1])
ax5.set_facecolor(PALETTE["panel"])
ax5.spines[:].set_color(PALETTE["grid"])
top15_s = sleeper_hits.head(15)
x = np.arange(len(top15_s))
w = 0.35

# 左轴：收藏加购用户数
ax5.bar(x - w/2, top15_s["fav_cart_users"], w,
        color=PALETTE["a3"], label="收藏/加购用户数", alpha=0.9)
ax5.set_ylabel("收藏/加购用户数", color=PALETTE["a3"])
ax5.tick_params(axis="y", colors=PALETTE["a3"])

# 右轴：购买用户数
ax5r = ax5.twinx()
ax5r.bar(x + w/2, top15_s["fav_cart_to_buy_users"], w,
         color=PALETTE["a1"], label="购买用户数", alpha=0.9)
ax5r.set_ylabel("购买用户数", color=PALETTE["a1"])
ax5r.tick_params(axis="y", colors=PALETTE["a1"])
ax5r.set_ylim(0, top15_s["fav_cart_to_buy_users"].max() * 5)
ax5r.patch.set_visible(False)
ax5r.spines[:].set_color(PALETTE["grid"])

ax5.set_xticks(x)
ax5.set_xticklabels(top15_s["item_id"].astype(str),
                    rotation=45, ha="right", fontsize=6, color=PALETTE["muted"])

# 合并图例
lines1, labels1 = ax5.get_legend_handles_labels()
lines2, labels2 = ax5r.get_legend_handles_labels()
ax5.legend(lines1 + lines2, labels1 + labels2,
           fontsize=8, labelcolor=PALETTE["muted"], framealpha=0)

ax5.tick_params(colors=PALETTE["muted"], labelsize=8)
ax5.grid(axis="y", color=PALETTE["grid"], linewidth=0.5, alpha=0.5)
ax5.set_title("⑤ 蓄力型爆款 Top15\n（收藏/加购用户 vs 购买用户 客户口径）",
              color=PALETTE["text"], fontsize=11, fontweight="bold", pad=10)

plt.savefig("../product_category_analysis.png", dpi=150,
            bbox_inches="tight", facecolor=PALETTE["bg"])
plt.show()
print("图表已保存：product_category_analysis.png")

In [27]:
print("=" * 65)
print("【关键洞察摘要】")

# ① GMV 集中度
print(f"\n① GMV 驱动集中度")
print(f"   Top5  品类购买占全平台: {top_gmv['buy'].head(5).sum()/total_buy:.1%}")
print(f"   Top10 品类购买占全平台: {top_gmv['buy'].head(10).sum()/total_buy:.1%}")
print(f"   核心驱动品类 ID: {', '.join(top_gmv['category_id'].head(5).astype(str).tolist())}")

# ② 转化率对比（次数 vs 客户）
print(f"\n② 加购→购买 转化率对比（价格敏感 vs 高意愿）")
print(f"   [次数口径] 价格敏感品类均值: {price_sensitive_cnt['cart_to_buy'].mean():.4f}")
print(f"   [次数口径] 高意愿品类均值:   {impulse_cnt['cart_to_buy'].mean():.4f}")
print(f"   [客户口径] 价格敏感品类均值: {price_sensitive_u['cart_to_buy_users_rate'].mean():.4f}")
print(f"   [客户口径] 高意愿品类均值:   {impulse_u['cart_to_buy_users_rate'].mean():.4f}")

# ③ fav+cart → buy 与纯 cart→buy 差异
all_cat_conv = cat_full[(cat_full["cart"] >= CART_MIN) & (cat_full["buy"] >= BUY_MIN)]
diff_mean = (all_cat_conv["cart_to_buy_users_rate"] - all_cat_conv["fav_cart_to_buy_users_rate"]).mean()
print(f"\n③ fav+cart 合并 vs 纯 cart 口径差异")
print(f"   合并后转化率平均低于纯 cart 口径: {diff_mean:.4f}")
print(f"   （说明加入 fav 分母后转化率下降，印证收藏行为购买意图偏低）")

# ④ 蓄力型爆款
print(f"\n④ 蓄力型爆款潜力品 Top5")
for _, row in sleeper_hits.head(5).iterrows():
    print(f"   item={row['item_id']:>10} | "
          f"fav={int(row['fav']):>5} cart={int(row['cart']):>5} buy={int(row['buy']):>4} | "
          f"热度分={int(row['heat_score']):>6} 潜力比率={row['potential_ratio']:.1f} | "
          f"收藏、加购用户={int(row['fav_cart_users']):>5} 购买用户={int(row['fav_cart_to_buy_users']):>4} "
          f"客户转化率={row['fav_cart_to_buy_rate']:.3f}")
print("=" * 65)

【关键洞察摘要】

① GMV 驱动集中度
   Top5  品类购买占全平台: 7.2%
   Top10 品类购买占全平台: 12.6%
   核心驱动品类 ID: 4756105, 4145813, 982926, 4801426, 2735466

② 加购→购买 转化率对比（价格敏感 vs 高意愿）
   [次数口径] 价格敏感品类均值: 0.0874
   [次数口径] 高意愿品类均值:   1.6064
   [客户口径] 价格敏感品类均值: 0.0494
   [客户口径] 高意愿品类均值:   0.3186

③ fav+cart 合并 vs 纯 cart 口径差异
   合并后转化率平均低于纯 cart 口径: 0.0115
   （说明加入 fav 分母后转化率下降，印证收藏行为购买意图偏低）

④ 蓄力型爆款潜力品 Top5
   item= 2671601.0 | fav=  158 cart=  173 buy=   1 | 热度分=   504 潜力比率=252.0 | 收藏、加购用户=  329 购买用户=   0 客户转化率=0.000
   item= 2161188.0 | fav=  233 cart=  117 buy=   1 | 热度分=   467 潜力比率=233.5 | 收藏、加购用户=  349 购买用户=   0 客户转化率=0.000
   item= 2678768.0 | fav=  194 cart=   88 buy=   1 | 热度分=   370 潜力比率=185.0 | 收藏、加购用户=  280 购买用户=   0 客户转化率=0.000
   item= 2642113.0 | fav=  214 cart=  230 buy=   3 | 热度分=   674 潜力比率=168.5 | 收藏、加购用户=  441 购买用户=   1 客户转化率=0.002
   item= 1283005.0 | fav=  199 cart=  190 buy=   3 | 热度分=   579 潜力比率=144.8 | 收藏、加购用户=  373 购买用户=   0 客户转化率=0.000
